## Pro

In [1]:
import sys

sys.path.append("/var/git/nec/projects")


In [2]:
from utils.generator import cotomi_pro_gen


/usr/local/lib/python3.10/site-packages/pydantic/_internal/_fields.py:160: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
""" This is cotomi-pro generator """

import json
from typing import Union, List, Optional, Dict
from pydantic.dataclasses import dataclass

import requests

from config import config


@dataclass
class ChoiceDelta:
    content: Optional[str] = None
    function_call: Optional[str] = None
    role: Optional[str] = None
    tool_calls: Optional[str] = None


@dataclass
class Choice:
    index: int
    delta: ChoiceDelta
    finish_reason: Optional[str] = None
    logprobs: Optional[str] = None
    content_filter_results: Optional[Dict[str, str]] = None


@dataclass
class ChatCompletionChunk:
    id: str
    object: str
    created: int
    model: str
    choices: List[Choice]


class StreamResponse():
    def __init__(self, ) -> None:
        self.full_text = ""

    def gen(self, message_list: list, **kwargs):
        """ generator of cotomi_pro
        - input:
            - message_list: [
                {"role": "assistant", "content": "what can i help you?"},
                {"role": "user", "content": "hello"},
            ],
            - kwargs: {
                "temperature": 0.1,
                "top_p": 0.95,
                ...
            }
        """

        url = config.cotomi_pro.base_url + '/chat/completions'

        payload = {
            "messages": message_list,
            "model": config.cotomi_pro.model_name,
            "stream": True,
            **kwargs,
        }

        headers = {
            'content-type': 'application/json',
        }

        response = requests.post(url,
                                 json=payload,
                                 headers=headers,
                                 stream=True,
                                 timeout=1000,
                                 )

        return response

    def __call__(self, messages: Union[str, List[str]], **params):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        print(messages)

        response = self.gen(messages, **params)

        for chunk in response.iter_content(chunk_size=1024):
            decoded_chunk = chunk.decode('utf-8')
            try:
                decoded_data = decoded_chunk.split('data: ')[1].strip()
                json_decoded_data = json.loads(decoded_data)
                data_chunk = ChatCompletionChunk(**json_decoded_data)
                yield data_chunk
            except IndexError:
                break
            except json.JSONDecodeError:
                break


gen = StreamResponse()
response = gen("こんにちは！")
full_text = ""

for chunk in response:
    print(chunk)
    try:
        content = chunk.choices[0].delta.content
        print(content)
        if content:
            full_text += content
    except KeyError:
        pass

print(full_text)


In [3]:
response = cotomi_pro_gen("Hello")

full_text = ""
for chunk in response:
    # print(chunk)
    try:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            full_text += content
    except KeyError:
        pass

print(full_text)


[{'role': 'user', 'content': 'Hello'}]
Hello! How can I assist you today? If you have any questions or need help with something, feel free to ask. I'm here to provide information and support on a wide range of topics, from general knowledge and research to programming and technology. Just let me know what you need. Hello! How can I assist you today? If you have any questions or need help with something, feel free to ask. I'm here to provide information and support on a wide range of topics, from general knowledge and research to programming and technology. Just let me know what you need. 


## Light

In [4]:
import sys

sys.path.append("/var/git/nec/projects")

params = {
    "temperature": 0.1,
    # range: 0.0 - 1.0
    "top_p": 0.95,
    # range: 1 -
    "n": 1,
    "stop": None,
    # range: 1 -
    "max_tokens": 1024,
}


In [5]:
from utils.generator import cotomi_light_gen

full_text = ""
response = cotomi_light_gen("マラソンで4位の人を抜きました。現在何位ですか？", **params)
# print(response)

for chunk in response:
    # print(chunk)
    try:
        if chunk:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="")
                full_text += content
    except KeyError:
        pass

print(full_text)


[{'role': 'user', 'content': 'マラソンで4位の人を抜きました。現在何位ですか？'}]
で4位の人を抜いたら、あなたはその時点で3位に浮上します。マラソンでは、通常、複数の選手が同じくらいの差で競い合っているため、何人かの人を抜くことで順位が上がる可能性があります。ただし、抜く人数や順位の変動については、レース展開によって異なります。現在の順位を正確に把握するためには、抜かれた人の順位やレースの状況詳細な情報が必要です。で4位の人を抜いたら、あなたはその時点で3位に浮上します。マラソンでは、通常、複数の選手が同じくらいの差で競い合っているため、何人かの人を抜くことで順位が上がる可能性があります。ただし、抜く人数や順位の変動については、レース展開によって異なります。現在の順位を正確に把握するためには、抜かれた人の順位やレースの状況詳細な情報が必要です。
